In [2]:
import os
import glob
import shutil
import numpy as np
import xarray as xr
import geopandas as gpd
import regionmask
import dask
from dask.diagnostics import ProgressBar
from dask.distributed import Client, LocalCluster

dask.config.set({"distributed.dashboard.link": "/proxy/{port}/status"})
cluster = LocalCluster(n_workers=64, threads_per_worker = 1)
client = Client(cluster)

# -----------------------
# Config
# -----------------------
NETID = "k16v981"

SST_DIR = f"/home/{NETID}/my_work/data/era5/era5_sst/"
FILES = sorted(glob.glob(os.path.join(SST_DIR, "era5_sst_*.nc")))

SHAPE_DIR = "../data/shapefiles"
BASIN_LAYERS = {
    # each shapefile has ONE polygon feature, so we just take geometry[0]
    "arabian_gulf": os.path.join(SHAPE_DIR, "ecoregions", "ecoregions.shp"),
    "gulf_oman": os.path.join(SHAPE_DIR, "iho", "iho.shp"),
    "red_and_aden": os.path.join(SHAPE_DIR, "provinces", "provinces.shp"),
}

OUT_DIR = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/basin_anoms"
os.makedirs(OUT_DIR, exist_ok=True)

TMP_DIR = os.path.join(OUT_DIR, "_tmp_basin_build")
os.makedirs(TMP_DIR, exist_ok=True)

START = "1950-01-01"
END   = "2025-12-31"

CLIM_START = "1991-01-01"
CLIM_END   = "2020-12-31"

PAD = 0.5  # degrees bbox padding

# -----------------------
# Helpers
# -----------------------
def _fix_time_and_expver(ds: xr.Dataset) -> xr.Dataset:
    if "valid_time" in ds.coords or "valid_time" in ds.dims:
        ds = ds.rename({"valid_time": "time"})

    if "expver" in ds.dims:
        if "expver" in ds.coords and np.any(ds["expver"].values == 1):
            ds = ds.sel(expver=1)
        else:
            ds = ds.isel(expver=0)
        ds = ds.drop_vars("expver", errors="ignore")

    if "expver" in ds.coords and "expver" not in ds.dims:
        ds = ds.drop_vars("expver", errors="ignore")

    return ds

def _pick_sst_var(ds: xr.Dataset) -> xr.DataArray:
    for v in ds.data_vars:
        nm = v.lower()
        if nm in ("sst", "sea_surface_temperature") or "sst" in nm or "sea_surface_temperature" in nm:
            return ds[v]
    return ds[list(ds.data_vars)[0]]

def _standardize_lon(da: xr.DataArray) -> xr.DataArray:
    lon_name = "longitude" if "longitude" in da.coords else "lon"
    lon = da[lon_name]
    if lon.max() > 180:
        lon_new = ((lon + 180) % 360) - 180
        da = da.assign_coords({lon_name: lon_new}).sortby(lon_name)
    return da

def _subset_bbox(da: xr.DataArray, bounds):
    lat_name = "latitude" if "latitude" in da.coords else "lat"
    lon_name = "longitude" if "longitude" in da.coords else "lon"

    minx, miny, maxx, maxy = bounds
    lat = da[lat_name]
    if lat[0] > lat[-1]:
        return da.sel({lat_name: slice(maxy, miny), lon_name: slice(minx, maxx)})
    else:
        return da.sel({lat_name: slice(miny, maxy), lon_name: slice(minx, maxx)})

def _make_mask(da_subset_2d: xr.DataArray, geom, basin_name: str) -> xr.DataArray:
    lat_name = "latitude" if "latitude" in da_subset_2d.coords else "lat"
    lon_name = "longitude" if "longitude" in da_subset_2d.coords else "lon"

    regions = regionmask.Regions([geom], names=[basin_name], abbrevs=[basin_name])
    mask = regions.mask(da_subset_2d[lon_name], da_subset_2d[lat_name])
    return mask  # 0 inside, NaN outside


def _save_year_piece_nc(ds_piece: xr.Dataset, tmp_dir: str, basin_name: str, src_file: str):
    out = os.path.join(tmp_dir, f"{basin_name}__{os.path.basename(src_file).replace('.nc','')}.nc")
    ds_piece.to_netcdf(out)
    return out


def _load_single_geom(shp_path: str):
    gdf = gpd.read_file(shp_path)

    if len(gdf) == 0:
        raise ValueError(f"{shp_path}: GeoDataFrame is empty (0 features). Check the shapefile/layer contents.")

    if gdf.crs is None:
        raise ValueError(f"{shp_path}: CRS missing (no .prj?).")

    gdf = gdf.to_crs("EPSG:4326")

    # If multiple features exist, union them; otherwise use the only feature
    geom = gdf.geometry.unary_union if len(gdf) > 1 else gdf.geometry.iloc[0]
    return geom


# -----------------------
# Main (NetCDF-temp version; NO zarr dependency)
# -----------------------
def build_basin_anoms(basin_name: str, shp_path: str):
    print(f"\n=== {basin_name} ===")
    geom = _load_single_geom(shp_path)

    minx, miny, maxx, maxy = geom.bounds
    bounds = (minx - PAD, miny - PAD, maxx + PAD, maxy + PAD)

    # where we’ll stash temporary masked pieces for this basin
    basin_tmp_dir = os.path.join(TMP_DIR, f"{basin_name}_pieces")
    os.makedirs(basin_tmp_dir, exist_ok=True)

    # clean old temp pieces (optional but keeps things sane)
    for old in glob.glob(os.path.join(basin_tmp_dir, "*.nc")):
        os.remove(old)

    tmp_paths = []  # IMPORTANT: define this inside the function

    # 1) Write masked SST "pieces" (NetCDF) for each input file/year
    for f in FILES:
        ds = xr.open_dataset(f)
        ds = _fix_time_and_expver(ds)

        sst = _pick_sst_var(ds)
        sst = _standardize_lon(sst)
        sst = sst.sel(time=slice(START, END))

        if sst.sizes.get("time", 0) == 0:
            ds.close()
            print(f"  - skipped {os.path.basename(f)} (no time overlap with {START}..{END})")
            continue

        sub = _subset_bbox(sst, bounds)

        lat_name = "latitude" if "latitude" in sub.coords else "lat"
        lon_name = "longitude" if "longitude" in sub.coords else "lon"

        if sub.sizes.get(lat_name, 0) == 0 or sub.sizes.get(lon_name, 0) == 0:
            ds.close()
            raise ValueError(
                f"{basin_name}: bbox subset returned empty grid for file {os.path.basename(f)}.\n"
                f"  bounds(minx,miny,maxx,maxy)={bounds}\n"
                f"  data lon range=({float(sst[lon_name].min())}, {float(sst[lon_name].max())})\n"
                f"  data lat range=({float(sst[lat_name].min())}, {float(sst[lat_name].max())})"
            )

        # compute mask on a single 2D slice (cheap)
        mask = _make_mask(sub.isel(time=0, drop=True), geom, basin_name)
        sub_masked = sub.where(mask == 0)

        ds_piece = xr.Dataset({"sst": sub_masked})

        # write this piece to NetCDF
        out_piece = os.path.join(
            basin_tmp_dir, f"{basin_name}__{os.path.basename(f).replace('.nc','')}.nc"
        )
        ds_piece.to_netcdf(out_piece)
        tmp_paths.append(out_piece)

        ds.close()
        print(f"  + wrote piece: {os.path.basename(out_piece)}")

    if not tmp_paths:
        raise RuntimeError(f"{basin_name}: no valid SST pieces were created. Check time range and files.")

    # 2) Open all pieces as one virtual dataset and compute anomalies
    ds_b = xr.open_mfdataset(
        tmp_paths,
        combine="by_coords",
        parallel=True,
        chunks="auto",
    )
    
    sst_b = ds_b["sst"]
    clim = sst_b.sel(time=slice(CLIM_START, CLIM_END)).groupby("time.month").mean("time")
    sst_anom = (sst_b.groupby("time.month") - clim).rename("sst_anom")
    
    print("▶ Computing SST anomalies...")
    with ProgressBar():
        sst_anom_c = sst_anom.astype("float32").load()
    
    ds_b.close()
    
    ds_out = xr.Dataset({"sst_anom": sst_anom_c})
    out_nc = os.path.join(OUT_DIR, f"era5_sst_anom_{basin_name}_1950_2025.nc")
    
    # 🔥 disable compression until this works reliably
    encoding = {"sst_anom": {"dtype": "float32"}}
    
    print("▶ Writing NetCDF...")
    with ProgressBar():
        ds_out.to_netcdf(out_nc, encoding=encoding)
    
    print(f"✅ wrote {out_nc}")


if __name__ == "__main__":
    if not FILES:
        raise FileNotFoundError(f"No SST files found: {SST_DIR}")

    for basin_name, shp_path in BASIN_LAYERS.items():
        build_basin_anoms(basin_name, shp_path)

    print("\n✅ All basins done.")



=== arabian_gulf ===
  - skipped era5_sst_1940.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1941.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1942.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1943.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1944.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1945.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1946.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1947.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1948.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1949.nc (no time overlap with 1950-01-01..2025-12-31)
  + wrote piece: arabian_gulf__era5_sst_1950.nc
  + wrote piece: arabian_gulf__era5_sst_1951.nc
  + wrote piece: arabian_gulf__era5_sst_1952.nc
  + wrote piece: arabian_gulf__era5_sst_1953.nc
  + wrote piece: arabian_gulf__era5_

ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/pr

▶ Computing SST anomalies...


/home/k16v981/.conda/envs/my_env/lib/python3.9/site-packages/distributed/client.py:3362: UserWarning: Sending large graph of size 24.85 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


▶ Writing NetCDF...
✅ wrote /home/k16v981/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/basin_anoms/era5_sst_anom_arabian_gulf_1950_2025.nc

=== gulf_oman ===
  - skipped era5_sst_1940.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1941.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1942.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1943.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1944.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1945.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1946.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1947.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1948.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1949.nc (no time overlap with 1950-01-01..2025-12-31)
  + wrote piece: gulf_oman__era5_sst_1950.nc
  + wrote piece: gulf_oman__era5_s

/home/k16v981/.conda/envs/my_env/lib/python3.9/site-packages/distributed/client.py:3362: UserWarning: Sending large graph of size 24.85 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


▶ Writing NetCDF...
✅ wrote /home/k16v981/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/basin_anoms/era5_sst_anom_gulf_oman_1950_2025.nc

=== red_and_aden ===
  - skipped era5_sst_1940.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1941.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1942.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1943.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1944.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1945.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1946.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1947.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1948.nc (no time overlap with 1950-01-01..2025-12-31)
  - skipped era5_sst_1949.nc (no time overlap with 1950-01-01..2025-12-31)
  + wrote piece: red_and_aden__era5_sst_1950.nc
  + wrote piece: red_and_aden__

/home/k16v981/.conda/envs/my_env/lib/python3.9/site-packages/distributed/client.py:3362: UserWarning: Sending large graph of size 24.85 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


▶ Writing NetCDF...
✅ wrote /home/k16v981/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/basin_anoms/era5_sst_anom_red_and_aden_1950_2025.nc

✅ All basins done.
